# Análisis exploratorio del dataset PRO-ACT
## Sistema de Ayuda a la Decisión para predicción de progresión en ELA
### Asignatura: Sistemas de Ayuda a la Decisión Clínica

## 1. Carga de librerías y datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficas
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ Librerías cargadas")

✅ Librerías cargadas


### Carga de datos

Se define la ruta al directorio con los archivos CSV del dataset PRO-ACT y se cargan los seis ficheros principales.

In [2]:
ruta = r"C:\Users\Paula\SAD_ELA\csv_proact/"

alsfrs   = pd.read_csv(ruta + "F_PROACT_ALSFRS.csv")
demo     = pd.read_csv(ruta + "F_PROACT_DEMOGRAPHICS.csv")
historia = pd.read_csv(ruta + "F_PROACT_ALSHISTORY.csv")
fvc      = pd.read_csv(ruta + "F_PROACT_FVC.csv")
vitals   = pd.read_csv(ruta + "F_PROACT_VITALSIGNS.csv", low_memory=False)
labs     = pd.read_csv(ruta + "F_PROACT_LABS.csv")

print("✅ Archivos cargados correctamente")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Paula\\SAD_ELA\\csv_proact/F_PROACT_ALSFRS.csv'

## 2. Estructura general de los datos

In [ ]:
archivos = {
    'ALSFRS': alsfrs,
    'Demographics': demo,
    'ALSHISTORY': historia,
    'FVC': fvc,
    'Vitals': vitals,
    'Labs': labs
}

for nombre, df in archivos.items():
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

## 3. Análisis de valores nulos
En datos clínicos reales es habitual encontrar valores ausentes. 
A continuación analizamos el porcentaje de nulos por columna en cada archivo.

In [ ]:
for nombre, df in archivos.items():
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(1)
    nulos_df = pd.DataFrame({'nulos': nulos, 'porcentaje': nulos_pct})
    nulos_df = nulos_df[nulos_df['nulos'] > 0]
    if len(nulos_df) > 0:
        print(f"\n=== {nombre} ===")
        print(nulos_df.to_string())

## 4. Exploración del archivo ALSFRS
El archivo ALSFRS contiene las mediciones longitudinales de la escala ALSFRS-R 
para cada paciente a lo largo del tiempo. Es el archivo central del proyecto.

In [ ]:
print("Número de pacientes únicos:", alsfrs['subject_id'].nunique())
print("Número de mediciones totales:", len(alsfrs))
print(f"\nMedia de mediciones por paciente: {len(alsfrs)/alsfrs['subject_id'].nunique():.1f}")
print("\nEstadísticas de ALSFRS_R_Total:")
print(alsfrs['ALSFRS_R_Total'].describe().round(2))

### 4.1 Distribución de la puntuación ALSFRS-R total

Se muestran la distribución de la puntuación total ALSFRS-R mediante un histograma y un boxplot comparativo por sexo, obtenido al unir los datos de ALSFRS con Demographics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histograma
mean_alsfrs = alsfrs['ALSFRS_R_Total'].mean()

axes[0].hist(alsfrs['ALSFRS_R_Total'].dropna(), bins=30, 
             color='steelblue', edgecolor='white')
axes[0].axvline(mean_alsfrs, color='darkred', linestyle='--', linewidth=2,
                label=f"Media: {mean_alsfrs:.2f}")
axes[0].legend(loc='upper left', frameon=True)
axes[0].set_title('Distribución de ALSFRS-R Total')
axes[0].set_xlabel('Puntuación ALSFRS-R (0-48)')
axes[0].set_ylabel('Frecuencia')

# Boxplot por sexo (uniendo con demographics)
alsfrs_demo = alsfrs.merge(
    demo[['subject_id', 'Sex']], on='subject_id', how='left'
)
alsfrs_demo_clean = alsfrs_demo.dropna(subset=['ALSFRS_R_Total', 'Sex'])
sns.boxplot(data=alsfrs_demo_clean, x='Sex', y='ALSFRS_R_Total', 
            ax=axes[1], palette='Set2', hue='Sex', legend=False)
axes[1].set_title('ALSFRS-R Total por sexo')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Puntuación ALSFRS-R')

plt.tight_layout()
plt.savefig(r"C:\Users\Paula\SAD_ELA\datos_procesados/graf_alsfrs_distribucion.png", 
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica guardada")

## 5. Exploración de datos demográficos

Analizamos la distribución de las dos variables demográficas principales: edad y sexo. Para la edad se representa el histograma con la línea de la mediana; para el sexo, un gráfico de barras con el recuento de pacientes.

In [ ]:
# ── Histograma de edad ──────────────────────────────────────────────────
median_age = demo['Age'].median()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(
    demo['Age'].dropna(),
    bins=25,
    color='#E07B39',
    edgecolor='white'
)
axes[0].axvline(
    median_age,
    color='darkred',
    linestyle='--',
    linewidth=2,
    label=f'Mediana: {median_age:.0f} años'
)
axes[0].legend(loc='upper left', frameon=True)
axes[0].set_title('Distribución de edad')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')

# ── Gráfico de barras por sexo ────────────────────────────────────────
sex_counts = demo['Sex'].value_counts().sort_index()
sex_labels = {1: 'Hombre', 0: 'Mujer'}
labels = [sex_labels.get(k, str(k)) for k in sex_counts.index]
colors  = ['#4472C4', '#ED7D31']

bars = axes[1].bar(labels, sex_counts.values, color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, sex_counts.values):
    pct = val / sex_counts.sum() * 100
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f"{val}({pct:.1f}%)",
        ha='center', va='bottom', fontsize=10
    )

axes[1].set_title('Distribución por sexo')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Número de pacientes')
axes[1].set_ylim(0, sex_counts.max() * 1.15)

plt.tight_layout()
plt.savefig(
    r"C:\Users\Paula\SAD_ELA\datos_procesados/graf_demo_distribucion.png",
    dpi=150,
    bbox_inches='tight'
)
plt.show()
print(f"Mediana de edad: {median_age:.0f} años")
print(f"Distribución por sexo: {sex_counts.rename(sex_labels)}")

print("✅ Gráfica guardada")

## 6. Conclusiones de la exploración
- El dataset PRO-ACT contiene mediciones longitudinales de más de 10.000 pacientes
- La escala ALSFRS-R presenta una distribución amplia, reflejando la heterogeneidad clínica de la ELA
- Existen valores nulos en varias columnas que deberán tratarse en el preprocesamiento
- La distribución por sexo muestra una mayor proporción de hombres, consistente con la epidemiología de la ELA
- Estos hallazgos motivarán las decisiones de preprocesamiento del notebook 02